In [1]:
# Cell 1: Import libraries and set file paths

from pathlib import Path
import re
import pandas as pd

RAW_FILE = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\raw\tabular\melbourne_crime_recorded_offences_2025.xlsx")

PROCESSED_DIR = Path(r"C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OFFENCES_CSV = PROCESSED_DIR / "crime_suburb_offences_clean.csv"
SUMMARY_CSV = PROCESSED_DIR / "crime_suburb_summary.csv"
WORKBOOK_OUT = PROCESSED_DIR / "melbourne_crime_cleaned_2025.xlsx"


In [2]:
# Cell 2: Helper functions

def snake_case(value):
    value = value.strip().lower()
    value = value.replace("/", "_")
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def clean_text(series):
    return (
        series.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def percentile_rank(series):
    if len(series) <= 1:
        return pd.Series([0.0] * len(series), index=series.index)

    return (series.rank(method="average") - 1) / (len(series) - 1)


def crime_label(score):
    if pd.isna(score):
        return "Unknown crime context"
    if score >= 66:
        return "Low crime context"
    if score >= 33:
        return "Moderate crime context"
    return "Higher crime context"


def crime_summary_text(row):
    largest_bucket = max(
        [
            ("personal-safety offences", row["person_offences"]),
            ("property-related offences", row["property_offences"]),
            ("public/drug/disorder offences", row["disorder_offences"]),
        ],
        key=lambda item: item[1],
    )[0]

    return (
        f"This suburb shows {row['crime_context_label'].lower()} compared with other suburbs "
        f"in the latest crime dataset. {largest_bucket.capitalize()} are the largest recorded "
        "component; use this as context, not as a judgement of the suburb."
    )


In [3]:
# Cell 3: Load Table 03 from the Excel file

crime_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Table 03",
    dtype={"Postcode": "string"}
)

crime_raw.head()


,Year,Year ending,Local Government Area,Postcode,Suburb/Town Name,Offence Division,Offence Subdivision,Offence Subgroup,Offence Count
0,2025,December,Alpine,3691,Dederang,B Property and deception offences,B10 Arson,B19 Other fire related offences,1
1,2025,December,Alpine,3691,Dederang,B Property and deception offences,B30 Burglary/Break and enter,B322 Non-residential non-aggravated burglary,2
2,2025,December,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B42 Steal from a motor vehicle,1
3,2025,December,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B49 Other theft,2
4,2025,December,Alpine,3691,Dederang,C Drug offences,C30 Drug use and possession,C32 Drug possession,1


In [4]:
# Cell 4: Standardise column names

crime = crime_raw.copy()
crime.columns = [snake_case(col) for col in crime.columns]

crime.columns


Index(['year', 'year_ending', 'local_government_area', 'postcode',
       'suburb_town_name', 'offence_division', 'offence_subdivision',
       'offence_subgroup', 'offence_count'],
      dtype='object')

In [5]:
# Cell 5: Keep only the recommended columns from the Safety data document

keep_columns = [
    "year",
    "year_ending",
    "local_government_area",
    "postcode",
    "suburb_town_name",
    "offence_division",
    "offence_subdivision",
    "offence_subgroup",
    "offence_count",
]

crime = crime[keep_columns].copy()

crime.head()


,year,year_ending,local_government_area,postcode,suburb_town_name,offence_division,offence_subdivision,offence_subgroup,offence_count
0,2025,December,Alpine,3691,Dederang,B Property and deception offences,B10 Arson,B19 Other fire related offences,1
1,2025,December,Alpine,3691,Dederang,B Property and deception offences,B30 Burglary/Break and enter,B322 Non-residential non-aggravated burglary,2
2,2025,December,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B42 Steal from a motor vehicle,1
3,2025,December,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B49 Other theft,2
4,2025,December,Alpine,3691,Dederang,C Drug offences,C30 Drug use and possession,C32 Drug possession,1


In [6]:
# Cell 6: Clean text fields, postcode, and offence count

text_columns = [
    "year_ending",
    "local_government_area",
    "suburb_town_name",
    "offence_division",
    "offence_subdivision",
    "offence_subgroup",
]

for column in text_columns:
    crime[column] = clean_text(crime[column])

crime["postcode"] = (
    crime["postcode"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(4)
)

crime["offence_count"] = pd.to_numeric(crime["offence_count"], errors="coerce")

crime.head()


,year,year_ending,local_government_area,postcode,suburb_town_name,offence_division,offence_subdivision,offence_subgroup,offence_count
0,2025,December,Alpine,3691,Dederang,B Property and deception offences,B10 Arson,B19 Other fire related offences,1
1,2025,December,Alpine,3691,Dederang,B Property and deception offences,B30 Burglary/Break and enter,B322 Non-residential non-aggravated burglary,2
2,2025,December,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B42 Steal from a motor vehicle,1
3,2025,December,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B49 Other theft,2
4,2025,December,Alpine,3691,Dederang,C Drug offences,C30 Drug use and possession,C32 Drug possession,1


In [7]:
# Cell 7: Remove unusable rows

crime = crime[crime["suburb_town_name"].notna()]
crime = crime[crime["offence_count"].notna()]
crime = crime[crime["offence_count"] > 0]

crime["offence_count"] = crime["offence_count"].astype("int64")

crime.shape


(370298, 9)

In [8]:
# Cell 8: Create matching keys for suburb and LGA

crime["suburb_key"] = crime["suburb_town_name"].str.upper().str.strip()
crime["lga_key"] = crime["local_government_area"].str.upper().str.strip()

crime.head()


,year,year_ending,local_government_area,postcode,suburb_town_name,offence_division,offence_subdivision,offence_subgroup,offence_count,suburb_key,lga_key
0,2025,December,Alpine,3691,Dederang,B Property and deception offences,B10 Arson,B19 Other fire related offences,1,DEDERANG,ALPINE
1,2025,December,Alpine,3691,Dederang,B Property and deception offences,B30 Burglary/Break and enter,B322 Non-residential non-aggravated burglary,2,DEDERANG,ALPINE
2,2025,December,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B42 Steal from a motor vehicle,1,DEDERANG,ALPINE
3,2025,December,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B49 Other theft,2,DEDERANG,ALPINE
4,2025,December,Alpine,3691,Dederang,C Drug offences,C30 Drug use and possession,C32 Drug possession,1,DEDERANG,ALPINE


In [9]:
# Cell 9: Deduplicate defensively

identity_columns = [
    "year",
    "year_ending",
    "local_government_area",
    "lga_key",
    "postcode",
    "suburb_town_name",
    "suburb_key",
    "offence_division",
    "offence_subdivision",
    "offence_subgroup",
]

crime_suburb_offences_clean = (
    crime.groupby(identity_columns, dropna=False, as_index=False)["offence_count"]
    .sum()
    .sort_values(["year", "suburb_key", "postcode", "offence_subgroup"])
    .reset_index(drop=True)
)

crime_suburb_offences_clean.insert(
    0,
    "crime_id",
    range(1, len(crime_suburb_offences_clean) + 1)
)

crime_suburb_offences_clean.head()


,crime_id,year,year_ending,local_government_area,lga_key,postcode,suburb_town_name,suburb_key,offence_division,offence_subdivision,offence_subgroup,offence_count
0,1,2016,December,Alpine,ALPINE,3737,Abbeyard,ABBEYARD,B Property and deception offences,B20 Property damage,B21 Criminal damage,1
1,2,2016,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,A211 FV Serious assault,7
2,3,2016,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,A212 Non-FV Serious assault,21
3,4,2016,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,"A22 Assault police, emergency services or othe...",3
4,5,2016,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,A231 FV Common assault,12


In [10]:
# Cell 10: Use the latest year for suburb summary

latest_year = crime_suburb_offences_clean["year"].max()

latest_crime = crime_suburb_offences_clean[
    crime_suburb_offences_clean["year"] == latest_year
].copy()

latest_year, latest_crime.shape


(2025, (38558, 12))

In [11]:
# Cell 11: Create crime buckets recommended in the Safety data document

latest_crime["person_offences"] = latest_crime["offence_count"].where(
    latest_crime["offence_division"].str.startswith("A ", na=False),
    0
)

latest_crime["property_offences"] = latest_crime["offence_count"].where(
    latest_crime["offence_division"].str.startswith("B ", na=False),
    0
)

latest_crime["disorder_offences"] = latest_crime["offence_count"].where(
    latest_crime["offence_division"].str.startswith(("C ", "D ", "E "), na=False),
    0
)

latest_crime["other_offences"] = latest_crime["offence_count"].where(
    latest_crime["offence_division"].str.startswith("F ", na=False),
    0
)

latest_crime.head()


,crime_id,year,year_ending,local_government_area,lga_key,postcode,suburb_town_name,suburb_key,offence_division,offence_subdivision,offence_subgroup,offence_count,person_offences,property_offences,disorder_offences,other_offences
331740,331741,2025,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,A211 FV Serious assault,10,10,0,0,0
331741,331742,2025,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,A212 Non-FV Serious assault,33,33,0,0,0
331742,331743,2025,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,"A22 Assault police, emergency services or othe...",4,4,0,0,0
331743,331744,2025,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,A231 FV Common assault,18,18,0,0,0
331744,331745,2025,December,Yarra,YARRA,3067,Abbotsford,ABBOTSFORD,A Crimes against the person,A20 Assault and related offences,A232 Non-FV Common assault,54,54,0,0,0


In [12]:
# Cell 12: Aggregate into one row per suburb/postcode

crime_suburb_summary = (
    latest_crime.groupby(["year", "suburb_key", "postcode"], as_index=False)
    .agg(
        suburb_name=("suburb_town_name", "first"),
        lga_key=("lga_key", lambda values: "; ".join(sorted(set(values.dropna())))),
        lga_name=("local_government_area", lambda values: "; ".join(sorted(set(values.dropna())))),
        total_offences=("offence_count", "sum"),
        person_offences=("person_offences", "sum"),
        property_offences=("property_offences", "sum"),
        disorder_offences=("disorder_offences", "sum"),
        other_offences=("other_offences", "sum"),
    )
    .sort_values(["suburb_key", "postcode"])
    .reset_index(drop=True)
)

crime_suburb_summary.head()


,year,suburb_key,postcode,suburb_name,lga_key,lga_name,total_offences,person_offences,property_offences,disorder_offences,other_offences
0,2025,ABBOTSFORD,3067,Abbotsford,YARRA,Yarra,1609,205,1130,273,1
1,2025,ABERFELDIE,3040,Aberfeldie,MOONEE VALLEY,Moonee Valley,169,26,102,41,0
2,2025,ACHERON,3714,Acheron,MURRINDINDI,Murrindindi,5,2,2,1,0
3,2025,ADAMS ESTATE,3984,Adams Estate,BASS COAST,Bass Coast,2,0,2,0,0
4,2025,ADDINGTON,3352,Addington,BALLARAT,Ballarat,1,0,1,0,0


In [21]:
# Cell 13: Calculate percentile-based crime context score
# Keep both a raw score and a user-facing 0-100 display score

crime_suburb_summary["total_offence_percentile"] = percentile_rank(
    crime_suburb_summary["total_offences"]
)

crime_suburb_summary["person_crime_percentile"] = percentile_rank(
    crime_suburb_summary["person_offences"]
)

crime_suburb_summary["property_crime_percentile"] = percentile_rank(
    crime_suburb_summary["property_offences"]
)

crime_suburb_summary["disorder_crime_percentile"] = percentile_rank(
    crime_suburb_summary["disorder_offences"]
)

# Raw weighted score from the model
crime_suburb_summary["crime_context_score_raw"] = (
    100
    - (
        0.5 * crime_suburb_summary["person_crime_percentile"]
        + 0.3 * crime_suburb_summary["property_crime_percentile"]
        + 0.2 * crime_suburb_summary["disorder_crime_percentile"]
    )
    * 100
).round(1)

# Rescale raw score to a full 0-100 display score
min_score = crime_suburb_summary["crime_context_score_raw"].min()
max_score = crime_suburb_summary["crime_context_score_raw"].max()

crime_suburb_summary["crime_context_score"] = (
    (crime_suburb_summary["crime_context_score_raw"] - min_score)
    / (max_score - min_score)
    * 100
).round(1)

crime_suburb_summary.head()


,year,suburb_key,postcode,suburb_name,lga_key,lga_name,total_offences,person_offences,property_offences,disorder_offences,other_offences,total_offence_percentile,person_crime_percentile,property_crime_percentile,disorder_crime_percentile,crime_context_score,crime_context_label,crime_context_summary,crime_context_score_raw
0,2025,ABBOTSFORD,3067,Abbotsford,YARRA,Yarra,1609,205,1130,273,1,0.954849,0.944398,0.964883,0.937709,5.7,Higher crime context,This suburb shows higher crime context compare...,5.1
1,2025,ABERFELDIE,3040,Aberfeldie,MOONEE VALLEY,Moonee Valley,169,26,102,41,0,0.786371,0.778010,0.797032,0.784908,24.2,Higher crime context,This suburb shows higher crime context compare...,21.5
2,2025,ACHERON,3714,Acheron,MURRINDINDI,Murrindindi,5,2,2,1,0,0.331940,0.464256,0.266095,0.347408,69.7,Low crime context,This suburb shows moderate crime context compa...,61.9
3,2025,ADAMS ESTATE,3984,Adams Estate,BASS COAST,Bass Coast,2,0,2,0,0,0.146739,0.145485,0.266095,0.141722,92.2,Low crime context,This suburb shows low crime context compared w...,81.9
4,2025,ADDINGTON,3352,Addington,BALLARAT,Ballarat,1,0,1,0,0,0.054557,0.145485,0.141931,0.141722,96.4,Low crime context,This suburb shows low crime context compared w...,85.6


In [22]:
# Cell 14: Add neutral labels and explanation text

percentile_columns = [
    "total_offence_percentile",
    "person_crime_percentile",
    "property_crime_percentile",
    "disorder_crime_percentile",
]

crime_suburb_summary[percentile_columns] = crime_suburb_summary[percentile_columns].round(4)

crime_suburb_summary["crime_context_label"] = crime_suburb_summary["crime_context_score"].apply(crime_label)

crime_suburb_summary["crime_context_summary"] = crime_suburb_summary.apply(
    crime_summary_text,
    axis=1
)

crime_suburb_summary.head()


,year,suburb_key,postcode,suburb_name,lga_key,lga_name,total_offences,person_offences,property_offences,disorder_offences,other_offences,total_offence_percentile,person_crime_percentile,property_crime_percentile,disorder_crime_percentile,crime_context_score,crime_context_label,crime_context_summary,crime_context_score_raw
0,2025,ABBOTSFORD,3067,Abbotsford,YARRA,Yarra,1609,205,1130,273,1,0.9548,0.9444,0.9649,0.9377,5.7,Higher crime context,This suburb shows higher crime context compare...,5.1
1,2025,ABERFELDIE,3040,Aberfeldie,MOONEE VALLEY,Moonee Valley,169,26,102,41,0,0.7864,0.7780,0.7970,0.7849,24.2,Higher crime context,This suburb shows higher crime context compare...,21.5
2,2025,ACHERON,3714,Acheron,MURRINDINDI,Murrindindi,5,2,2,1,0,0.3319,0.4643,0.2661,0.3474,69.7,Low crime context,This suburb shows low crime context compared w...,61.9
3,2025,ADAMS ESTATE,3984,Adams Estate,BASS COAST,Bass Coast,2,0,2,0,0,0.1467,0.1455,0.2661,0.1417,92.2,Low crime context,This suburb shows low crime context compared w...,81.9
4,2025,ADDINGTON,3352,Addington,BALLARAT,Ballarat,1,0,1,0,0,0.0546,0.1455,0.1419,0.1417,96.4,Low crime context,This suburb shows low crime context compared w...,85.6


In [23]:
# Cell 15: Quick validation checks

print("Clean offence rows:", len(crime_suburb_offences_clean))
print("Summary rows:", len(crime_suburb_summary))
print("Latest year:", latest_year)

print(
    "Raw crime score range:",
    crime_suburb_summary["crime_context_score_raw"].min(),
    "to",
    crime_suburb_summary["crime_context_score_raw"].max()
)

print(
    "Display crime score range:",
    crime_suburb_summary["crime_context_score"].min(),
    "to",
    crime_suburb_summary["crime_context_score"].max()
)

crime_suburb_summary["crime_context_label"].value_counts()


Clean offence rows: 370298
Summary rows: 2393
Latest year: 2025
Raw crime score range: 0.0 to 88.8
Display crime score range: 0.0 to 100.0


crime_context_label
Low crime context         1095
Higher crime context       674
Moderate crime context     624
Name: count, dtype: int64

In [24]:
# Cell 16: Save cleaned CSV files

crime_suburb_offences_clean.to_csv(OFFENCES_CSV, index=False)
crime_suburb_summary.to_csv(SUMMARY_CSV, index=False)

print("Saved:")
print(OFFENCES_CSV)
print(SUMMARY_CSV)


Saved:
C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\crime_suburb_offences_clean.csv
C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\crime_suburb_summary.csv
